# Weekly FPL analysis

Thin notebook: all logic lives in `src/fpl`. This just calls the pipeline and inspects results.

In [ ]:
from fpl import db

db.init_db()

## 1. Update data (scrape + references + stats)

Equivalent to running `python scripts/update_data.py`.

In [ ]:
from fpl.scrape import understat
from fpl.reference import refresh as refresh_references
from fpl.stats import player_stats, team_gsxg

understat.run()
refresh_references.run()
player_stats.run()
team_gsxg.run()

In [ ]:
gsxg = db.read_table("gsxg")
gsxg_avg = db.read_table("gsxg_avg")
player_stats = db.read_table("understat_player_stats")

gsxg['team'].unique()

In [ ]:
gsxg[gsxg['team'] == 'Manchester City']

In [ ]:
gsxg_team_avg = gsxg[['team', 'xg_factor', 'xgc_factor']].groupby('team').mean()
gsxg_team_avg.sort_values(by='xg_factor', ascending=False)

In [ ]:
gsxg_team_avg.sort_values(by='xgc_factor', ascending=True)

In [ ]:
gsxg_avg

## 2. Simulate upcoming gameweeks

In [ ]:
from fpl.simulate import engine

N_SIMULATIONS = 100  # increase for more stable estimates, at the cost of runtime
N_GAMEWEEKS = 5

results = engine.simulate_next_gameweeks(n_simulations=N_SIMULATIONS, n_gameweeks=N_GAMEWEEKS)
gameweek_ids = sorted(results)
gameweek_ids

### Sense-check: top predicted scorers per gameweek

In [ ]:
gw_stats, scorelines = results[gameweek_ids[0]]
gw_stats.sort_values("points", ascending=False).head(15)

In [ ]:
for gw_id in gameweek_ids:
    print(f"\nGameweek {gw_id}")
    display(results[gw_id][0].sort_values("points", ascending=False).head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def scoreline_heatmap(scorelines, home_team, away_team, max_goals=5):
    """Heatmap of P(home_goals, away_goals) for one fixture.

    scorelines: the dict returned alongside gw_stats from simulate_next_gameweeks,
    keyed by (home_team, away_team) -> {(home_goals, away_goals): probability}.
    Scores above max_goals are bucketed into a "N+" row/column so the
    displayed probabilities still sum to (close to) 100%.
    """
    probs = scorelines[(home_team, away_team)]

    df = pd.Series(probs).rename_axis(["home_goals", "away_goals"]).reset_index(name="prob")
    df["home_goals"] = df["home_goals"].clip(upper=max_goals)
    df["away_goals"] = df["away_goals"].clip(upper=max_goals)
    grid = df.groupby(["home_goals", "away_goals"])["prob"].sum().unstack(fill_value=0)
    grid = grid.reindex(index=range(max_goals + 1), columns=range(max_goals + 1), fill_value=0)

    labels = [str(i) for i in range(max_goals)] + [f"{max_goals}+"]
    grid.index = labels
    grid.columns = labels

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(grid, annot=True, fmt=".1%", cmap="viridis",
                cbar_kws={"label": "Probability"}, ax=ax)
    ax.set_xlabel(f"{away_team} goals")
    ax.set_ylabel(f"{home_team} goals")
    ax.set_title(f"{home_team} vs {away_team} — scoreline probabilities\n"
                 f"(covers {df['prob'].sum():.1%} of simulated outcomes)")
    ax.invert_yaxis()  # 0 goals at the bottom, like a normal grid
    
    # Calculate result probabilities
    home_win_prob = sum(prob for (h, a), prob in probs.items() if h > a)
    draw_prob = sum(prob for (h, a), prob in probs.items() if h == a)
    away_win_prob = sum(prob for (h, a), prob in probs.items() if h < a)
    
    probabilities = {
        "home_win": home_win_prob,
        "draw": draw_prob,
        "away_win": away_win_prob
    }
    
    return ax, probabilities

In [ ]:
gw_stats, scorelines = results[gameweek_ids[1]]

# check the exact team-name strings used as keys before calling the function
list(scorelines.keys())

In [ ]:
# Cell — plot one fixture
home_team = 'Man Utd'
away_team = 'Man City'

ax, probabilities = scoreline_heatmap(scorelines, home_team, away_team, max_goals=5)

probabilities

## 3. Optimise squad

`fpl.squad.optimize` builds each player's expected points as a weighted
average across `N_GAMEWEEKS` (nearer gameweeks weighted more heavily —
see `optimize.GAMEWEEK_WEIGHTS`), then solves for the best squad under
FPL's budget/position/club constraints via `fpl.squad.optimize.select_squad`.

This picks a squad from scratch. If you're planning transfers for a squad
you already own, skip to section 4 instead.

In [ ]:
from fpl.squad import optimize

# optimize.GAMEWEEK_WEIGHTS defaults to [10, 9, 8, 7, 6] (len must match
# N_GAMEWEEKS above); trim/extend it if you changed N_GAMEWEEKS.
weights = optimize.GAMEWEEK_WEIGHTS[:N_GAMEWEEKS]

player_pool = optimize.build_player_pool(results, gameweek_ids=gameweek_ids, weights=weights)

# Exclude newly promoted teams
promoted_teams = ["Hull City", "Coventry City", "Ipswich Town"]
player_pool = player_pool[~player_pool['team'].isin(promoted_teams)]
player_pool.sort_values("expected_points", ascending=False).head(10)

### Best squad from scratch

In [ ]:
squad = optimize.select_squad(player_pool, expected_points_col="expected_points")

print(f"Total cost: {squad['now_cost'].sum():.1f}")
print(f"Total weighted expected points: {squad['expected_points'].sum():.2f}")
display(squad.sort_values(["position", "expected_points"], ascending=[True, False]))

### Best starting XI + captain per gameweek

`evaluate_squad` needs one points column per gameweek (not the single
blended `expected_points` column above), so pull those straight out of
`results` for just the squad we picked.

In [ ]:
from fpl.squad.evaluate import evaluate_squad

points_columns = [f"GW {gw_id} points" for gw_id in gameweek_ids]
squad_gw_points = squad.copy()
for gw_id, col in zip(gameweek_ids, points_columns):
    squad_gw_points[col] = results[gw_id][0].reindex(squad.index)["points"].fillna(0.0)

total_weighted_points, lineups = evaluate_squad(squad_gw_points, points_columns, weights)

print(f"Total weighted points across {len(gameweek_ids)} gameweeks: {total_weighted_points:.2f}")
for week_number, gw_id in enumerate(gameweek_ids, start=1):
    lineup = lineups[week_number]
    print(f"\nGameweek {gw_id} — {lineup['gw_points']:.2f} pts, captain: {lineup['captain']}")
    print(", ".join(lineup["lineup"]))

## 4. Plan transfers for your existing squad

Fetches your current squad and finds the best one reachable within
`MAX_TRANSFERS` changes, using `fpl.squad.optimize.plan_transfers`. This
models real transfer economics — keeping a player costs nothing, each
outgoing player refunds its sale price, each incoming player costs its
market price, and total spend can't exceed bank + refunds.

If `FPL_EMAIL`/`FPL_PASSWORD` are set in `.env`, this authenticates via
`fpl.squad.my_team` to get your *real* sale prices and bank balance
(FPL's sell-on fee means sale price can trail market price). Otherwise
it falls back to the public picks endpoint, approximates sale price
from current market price, and treats bank as £0. Skip this section if
you'd rather just look at the from-scratch squad above.

In [ ]:
from fpl.squad import my_team

try:
    picks_df, transfer_status = my_team.fetch_my_team()
    id_to_name = {row.fpl_id: name for name, row in player_pool.iterrows()}
    current_squad = [id_to_name[fpl_id] for fpl_id in picks_df.index]
    bank = transfer_status["bank"]

    # Attach real sale prices to player_pool (only meaningful for
    # current-squad players -- everyone else keeps their market price).
    player_pool["selling_price"] = player_pool["now_cost"]
    fpl_id_to_selling_price = picks_df["selling_price"].to_dict()
    for name in current_squad:
        player_pool.loc[name, "selling_price"] = fpl_id_to_selling_price[player_pool.loc[name, "fpl_id"]]

    print(
        f"Authenticated via FPL_EMAIL/FPL_PASSWORD — bank £{bank:.1f}m, "
        f"{transfer_status['free_transfers']} free transfer(s), "
        f"squad sale value £{player_pool.loc[current_squad, 'selling_price'].sum():.1f}m "
        "(real sell-on prices, not market prices)."
    )
except ValueError:
    # FPL_EMAIL/FPL_PASSWORD not set in .env -- fall back to the public
    # picks endpoint. Sale price is then approximated as current market
    # price, which can overstate it slightly for any player who's risen
    # in value since you bought them.
    import requests

    from fpl.config import FPL_ENTRY_ID, FPL_ENTRY_URL_TEMPLATE, FPL_PICKS_URL_TEMPLATE

    if not FPL_ENTRY_ID:
        raise ValueError("Set FPL_ENTRY_ID in .env to fetch your current squad (see .env.example)")

    entry = requests.get(FPL_ENTRY_URL_TEMPLATE.format(entry_id=FPL_ENTRY_ID), timeout=30).json()
    current_event = entry["current_event"]
    picks_url = FPL_PICKS_URL_TEMPLATE.format(entry_id=FPL_ENTRY_ID, event_id=current_event)
    picks = requests.get(picks_url, timeout=30).json()["picks"]

    id_to_name = {row.fpl_id: name for name, row in player_pool.iterrows()}
    current_squad = [id_to_name[pick["element"]] for pick in picks]
    bank = 0.0  # unknown without authenticating -- treat as 0 rather than guess
    player_pool["selling_price"] = player_pool["now_cost"]  # approximation, see caveat above
    print(
        "No FPL_EMAIL/FPL_PASSWORD in .env — approximating sale price from current market price "
        "and treating bank as £0.0m. Set them to use fpl.squad.my_team for exact figures."
    )

current_squad

In [ ]:
MAX_TRANSFERS = 2  # None for unlimited; free hits aside, more transfers cost 4pts each in FPL

new_squad = optimize.plan_transfers(
    player_pool,
    current_squad=current_squad,
    expected_points_col="expected_points",
    bank=bank,
    sale_price_col="selling_price",
    max_transfers=MAX_TRANSFERS,
)

players_out = [name for name in current_squad if name not in new_squad.index]
players_in = [name for name in new_squad.index if name not in current_squad]

if not players_out:
    print("No transfer recommended — current squad is already the best affordable option.")
else:
    for out_name, in_name in zip(players_out, players_in):
        print(f"OUT: {out_name}  ->  IN: {in_name}")

old_points = player_pool.loc[current_squad, "expected_points"].sum()
print(f"\nOld total weighted expected points: {old_points:.2f}")
print(f"New total weighted expected points: {new_squad['expected_points'].sum():.2f}")